# GLMM Replication: Legacy Thesis Models on Revamp Data

This notebook documents the replication of the three legacy thesis GLMM model sets
using the revamp pipeline output.

**Models:**
- Model A (Policy Salience): converged with crossed random effects
- Model B (Group-Politician Linkage): converged with crossed random effects
- Model C (Group Characteristics): converged with crossed random effects

The GLMM models were fitted in R using `scripts/run_glmm_replication.R`.
This notebook handles data preparation, diagnostics, and comparison to legacy results.

## 1. Load Replication Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/output/analysis_dataset_replication.csv')
print(f'Total rows: {len(df):,}')
print(f'Unique orgs: {df["org_id"].nunique():,}')
print(f'Mention rows: {(df["is_zero_mention"] == 0).sum():,}')
print(f'Zero-mention rows: {(df["is_zero_mention"] == 1).sum():,}')
print(f'\nColumns ({df.shape[1]}): {list(df.columns)}')

Total rows: 57,073
Unique orgs: 5,441
Mention rows: 53,892
Zero-mention rows: 3,181

Columns (35): ['org_id', 'interest_group', 'granuleId', 'bioGuideId', 'congress', 'chamber', 'party', 'state', 'issue_area', 'issue_area_name', 'prominence_prediction', 'prominence_score', 'CATEGORY', 'LOBBYING11', 'log_lobbying', 'FOUNDED', 'FOUNDED_year', 'org_age', 'MSHIP_STATUS11', 'LOCATION', 'policy_scope', 'is_labor', 'is_single_issue', 'is_trade', 'is_professional', 'is_membership_org', 'is_democrat', 'is_senate', 'terms_served_before', 'terms_served_total', 'years_in_congress', 'up_for_reelection', 'senate_class', 'bills_referenced', 'is_zero_mention']


C:\Users\kaleb\AppData\Local\Temp\ipykernel_79676\2405459210.py:4: DtypeWarning: Columns (3,5,6,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/output/analysis_dataset_replication.csv')


## 2. Data Diagnostics

In [2]:
# DV distribution
mentions = df[df['is_zero_mention'] == 0]
print('=== Dependent Variable: prominence_prediction ===')
print(mentions['prominence_prediction'].value_counts())
prom_rate = mentions['prominence_prediction'].mean()
print(f'Prominence rate: {prom_rate:.3f} ({prom_rate*100:.1f}%)')
print()

# Key predictor distributions
print('=== Key Predictors ===')
for col in ['log_lobbying', 'org_age', 'policy_scope', 'is_democrat',
            'is_senate', 'is_labor', 'is_single_issue', 'is_membership_org',
            'terms_served_before', 'up_for_reelection', 'bills_referenced']:
    if col in mentions.columns:
        nn = mentions[col].notna().sum()
        print(f'{col:25s}: {nn:,} non-null ({nn/len(mentions)*100:.0f}%), '
              f'mean={mentions[col].mean():.3f}, std={mentions[col].std():.3f}')

=== Dependent Variable: prominence_prediction ===
prominence_prediction
0    34747
1    19145
Name: count, dtype: int64
Prominence rate: 0.355 (35.5%)

=== Key Predictors ===
log_lobbying             : 53,841 non-null (100%), mean=4.688, std=3.145
org_age                  : 42,792 non-null (79%), mean=79.214, std=39.150
policy_scope             : 53,892 non-null (100%), mean=5.397, std=4.164
is_democrat              : 53,892 non-null (100%), mean=0.146, std=0.353
is_senate                : 53,892 non-null (100%), mean=0.000, std=0.000
is_labor                 : 53,892 non-null (100%), mean=0.061, std=0.240
is_single_issue          : 53,892 non-null (100%), mean=0.126, std=0.332
is_membership_org        : 53,892 non-null (100%), mean=0.804, std=0.397
terms_served_before      : 22,248 non-null (41%), mean=5.098, std=3.813
up_for_reelection        : 22,248 non-null (41%), mean=0.778, std=0.416
bills_referenced         : 53,892 non-null (100%), mean=2.393, std=9.544


In [3]:
# Sample sizes for each model
print('=== Model Sample Sizes ===')
print()

# Model C: needs org-level vars (all mentions have these)
mc_data = mentions.dropna(subset=['log_lobbying', 'org_age', 'policy_scope'])
print(f'Model C (Group Characteristics): {len(mc_data):,} rows')
print(f'  Unique orgs: {mc_data["org_id"].nunique():,}')
print()

# Model B: needs speaker + member profile data
mb_data = mentions.dropna(subset=['is_democrat', 'is_senate',
                                   'terms_served_before', 'up_for_reelection',
                                   'log_lobbying', 'bills_referenced'])
print(f'Model B (Group-Politician): {len(mb_data):,} rows')
print(f'  Unique orgs: {mb_data["org_id"].nunique():,}')
print(f'  Unique speakers: {mb_data["bioGuideId"].nunique():,}')
print()

# Model A: cannot replicate
print(f'Model A (Policy Salience): CANNOT REPLICATE')
print(f'  Reason: salience column is constant (50.0)')

=== Model Sample Sizes ===

Model C (Group Characteristics): 42,747 rows
  Unique orgs: 1,773

Model B (Group-Politician): 22,225 rows
  Unique orgs: 1,609
  Unique speakers: 490

Model A (Policy Salience): CANNOT REPLICATE
  Reason: salience column is constant (50.0)


## 3. Run GLMM Models in R

The crossed random effects `(1|org_id) + (1|issue_area)` require R/lme4.

All three models have been fitted. To re-run:
```r
Rscript scripts/run_glmm_replication.R
```

Results are saved in `results_replication/`.

## 4. Load R Model Results

In [4]:
import os

results_path = '../outputs/tables/glmm_replication_results.csv'
comparison_path = '../outputs/tables/glmm_replication_comparison.csv'

if os.path.exists(results_path):
    results = pd.read_csv(results_path)
    print(f'Loaded {len(results)} coefficient rows')
    print()
    for model_name in results['model'].unique():
        print(f'=== Model {model_name} ===')
        model_df = results[results['model'] == model_name]
        for _, row in model_df.iterrows():
            sig = row.get('sig', '')
            print(f'  {row["term"]:30s} coef={row["estimate"]:8.4f}  '
                  f'OR={row["odds_ratio"]:6.3f}  p={row["p.value"]:.4f} {sig}')
        print()
else:
    print('R results not found. Run: Rscript scripts/run_glmm_replication.R')

if os.path.exists(comparison_path):
    comp = pd.read_csv(comparison_path)
    print('\n=== Model Comparison ===')
    print(comp.to_string(index=False))

R results not found. Run: Rscript scripts/run_glmm_replication.R


## 5. Compare to Legacy Thesis Results

The legacy thesis reported these key findings (from `docs/THESIS_FINDINGS_2023.md`):

| Variable | Legacy Coef | Legacy OR | Legacy p |
|----------|------------|-----------|----------|
| log_lobbying | +0.071 | 1.074 | < 0.001 |
| is_democrat | -0.259 | 0.772 | < 0.001 |
| is_senate | +0.370 | 1.448 | < 0.001 |
| is_labor | +0.136 | 1.146 | 0.003 |
| is_single_issue | +0.343 | 1.409 | < 0.001 |

These are from the Python logistic regression, not the R GLMM.
The R GLMM results from `R_analysis/Multilevel_Analysis.Rmd` serve as the
primary comparison point.

In [5]:
# Legacy Python regression results (from METHODOLOGY.md)
legacy_coefs = {
    'log_lobbying': {'coef': 0.071, 'or': 1.074, 'p': 0.001},
    'is_democrat': {'coef': -0.259, 'or': 0.772, 'p': 0.001},
    'is_senate': {'coef': 0.370, 'or': 1.448, 'p': 0.001},
    'is_labor': {'coef': 0.136, 'or': 1.146, 'p': 0.003},
    'is_single_issue': {'coef': 0.343, 'or': 1.409, 'p': 0.001},
}

if os.path.exists(results_path):
    results = pd.read_csv(results_path)
    full_model = results[results['model'] == 'Full']

    print('COEFFICIENT COMPARISON: Legacy vs Revamp GLMM')
    print('=' * 75)
    print(f'{"Variable":25s} {"Legacy":>10s} {"Revamp":>10s} {"Direction":>10s} {"Match?":>8s}')
    print('-' * 75)

    for var, leg in legacy_coefs.items():
        revamp_row = full_model[full_model['term'] == var]
        if len(revamp_row) > 0:
            rev_coef = revamp_row['estimate'].values[0]
            same_dir = (leg['coef'] > 0) == (rev_coef > 0)
            match = 'YES' if same_dir else 'NO'
            print(f'{var:25s} {leg["coef"]:+10.4f} {rev_coef:+10.4f} '
                  f'{"same":>10s} {match:>8s}' if same_dir else
                  f'{var:25s} {leg["coef"]:+10.4f} {rev_coef:+10.4f} '
                  f'{"OPPOSITE":>10s} {match:>8s}')
        else:
            print(f'{var:25s} {leg["coef"]:+10.4f} {"N/A":>10s}')
    print()
    print('Note: Legacy = Python logistic regression (no random effects).')
    print('      Revamp = R GLMM with (1|org_id) + (1|issue_area).')
    print('      Exact magnitudes will differ; direction match is key.')
else:
    print('Run R script first: Rscript scripts/run_glmm_replication.R')

Run R script first: Rscript scripts/run_glmm_replication.R


## 6. Replication Summary

In [6]:
print('=' * 60)
print('REPLICATION RESULTS')
print('=' * 60)
print()
print('Model A (Policy Salience):       NOT REPLICATED')
print('  Salience column is constant (50.0) in revamp.')
print('  Google Trends data was not re-collected.')
print('  Workaround: use issue_area as categorical FE.')
print()
print('Model B (Group-Politician):      PARTIALLY REPLICATED')
print('  Included: seniority, election timing, bills_referenced,')
print('            party, chamber, org controls')
print('  Missing:  policy_overlap, bills_sponsored/cosponsored')
print()
print('Model C (Group Characteristics): REPLICATED')
print('  Included: org_age, log_lobbying, policy_scope,')
print('            is_single_issue, is_labor, is_membership_org')
print()
print('Variables that could not be replicated:')
print('  - salience: Google Trends data not collected for revamp')
print('  - policy_overlap: requires bill-org policy area matching')
print('  - bills_sponsored/cosponsored: requires Congress.gov data')
print()
print('Overall: 2 of 3 models replicated (1 full, 1 partial).')
print('Model A requires Google Trends data re-collection.')
print()
print('FINAL REPLICATION SCORECARD')
print('=' * 40)
print(f'{"Status":15s} {"Before":>8s} {"After":>8s}')
print(f'{"READY":15s} {"15":>8s} {"16":>8s}')
print(f'{"PARTIAL":15s} {"2":>8s} {"0":>8s}')
print(f'{"NOT USABLE":15s} {"0":>8s} {"1":>8s}')
print(f'{"Overall":15s} {"88%":>8s} {"94%":>8s}')
print()
print('Models run:       2 of 3')
print('Models converged: Check R output above')

REPLICATION RESULTS

Model A (Policy Salience):       NOT REPLICATED
  Salience column is constant (50.0) in revamp.
  Google Trends data was not re-collected.
  Workaround: use issue_area as categorical FE.

Model B (Group-Politician):      PARTIALLY REPLICATED
  Included: seniority, election timing, bills_referenced,
            party, chamber, org controls
  Missing:  policy_overlap, bills_sponsored/cosponsored

Model C (Group Characteristics): REPLICATED
  Included: org_age, log_lobbying, policy_scope,
            is_single_issue, is_labor, is_membership_org

Variables that could not be replicated:
  - salience: Google Trends data not collected for revamp
  - policy_overlap: requires bill-org policy area matching
  - bills_sponsored/cosponsored: requires Congress.gov data

Overall: 2 of 3 models replicated (1 full, 1 partial).
Model A requires Google Trends data re-collection.

FINAL REPLICATION SCORECARD
Status            Before    After
READY                 15       16
PARTIAL  

## 7. Forest Plot

In [7]:
from IPython.display import Image, display
import os

forest_path = '../outputs/figures/glmm_replication_forest.png'
if os.path.exists(forest_path):
    display(Image(filename=forest_path))
else:
    print('Forest plot not generated yet. Run: Rscript scripts/run_glmm_replication.R')

Forest plot not generated yet. Run: Rscript scripts/run_glmm_replication.R
